In [1]:
import sys
! {sys.executable} -m pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import logging 
import os
import pandas as pd
import mysql.connector
from mysql.connector import Error as MySQLError

In [ ]:
#sample data
#order_id,order_line_item,order_date,customer_email,customer_city,store_code,product_sku,category,quantity,unit_price,store_discount,#payment_gateway_discount,tax_amount,amount_paid,transaction_type
#ORD-10001,1,2021-11-16,customer_1@email.com,Seattle,STORE-SEA04,SKU-H301,Home,2,141.01,0.0,0.0,22.56,304.58,SALE
#ORD-10002,1,2025-06-30,customer_2@email.com,Miami,STORE-MIA05,SKU-H301,Home,1,53.49,2.67,0.0,4.07,54.89,SALE
#ORD-10003,1,2023-04-18,customer_3@email.com,Chicago,STORE-CHI02,SKU-A202,Apparel,3,106.39,31.92,0.0,22.98,310.23,SALE
#ORD-10004,1,2022-10-17,customer_4@email.com,Austin,STORE-AUS07,SKU-E101,Electronics,2,659.74,0.0,0.0,105.56,1425.04,SALE
#ORD-10005,1,2025-03-31,customer_5@email.com,Chicago,STORE-CHI02,SKU-H301,Home,4,131.85,0.0,0.0,42.19,569.59,SALE

In [3]:
#variables
db_name=os.getenv("DB_NAME","retail_warehouse")
db_host=os.getenv("DB_HOST","localhost")
db_user=os.getenv("DB_USER","root")
db_port=os.getenv("DB_PORT",3306)
db_pass=os.getenv("DB_PASS")



In [4]:
def get_db_connection():
    conn=mysql.connector.connect(
        host=db_host,
        database=db_name,
        password=db_pass,
        user=db_user,
        port=int(db_port),
        charset="utf8mb4",      
        autocommit=False
    )
    return conn

In [5]:
def ensure_database():
    try:
        conn = mysql.connector.connect(host=db_host, user=db_user, password=db_pass, port=int(db_port))
        cur = conn.cursor()
        cur.execute(f"CREATE DATABASE IF NOT EXISTS `{db_name}` CHARACTER SET utf8mb4")
        cur.close()
        conn.close()
    except MySQLError as e:     
        log.warning("Could not create database '%s' (continuing): %s", db_name, e)

In [6]:
reject_file="rejected_rows.csv"
suspect_file="suspect_rows.csv"


In [7]:
BATCH_SIZE=10_000
AMOUNT_TOLERANCE=0.05


In [8]:
COLUMNS=["order_id", "order_line_item", "order_date", "customer_email", "customer_city",
    "store_code", "product_sku", "category", "quantity", "unit_price",
    "store_discount", "payment_gateway_discount", "tax_amount", "amount_paid",
    "transaction_type"]
INT_COLS = ["order_line_item", "quantity"]
MONEY_COLS = ["unit_price", "store_discount", "payment_gateway_discount", "tax_amount", "amount_paid"]
ZERO_FILL_COLS = ["store_discount", "payment_gateway_discount", "tax_amount"] 
NULL_LIKE = {"", "nan", "null", "none", "n/a", "na", "-"}
EMAIL_REGEX=r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
MAX_LEN = {"order_id": 50, "customer_email": 255, "customer_city": 100, "store_code": 50,
           "product_sku": 50, "category": 50, "transaction_type": 20}


In [9]:
logging.basicConfig(level=logging.INFO,format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("etl_pipeline.log", encoding="utf-8")
    ],force=True)

log=logging.getLogger("retail_etl")

In [10]:
# extraction part
def extract_data(file_path):
    df=pd.read_csv(file_path,dtype=str,keep_default_na=False,encoding="utf-8-sig",on_bad_lines="warn",skipinitialspace=True)
    df.columns=df.columns.str.strip().str.lower()
    missing_columns=[c for c in COLUMNS if c not in df.columns]
    if missing_columns:
         raise ValueError(f"Source file is missing expected columns: {missing_columns}")
    log.info("Extracted %d rows", len(df))
    return df[COLUMNS]   



In [11]:
def clean_text(series):
    series=series.astype("string").str.strip().str.lower()
    return series.mask(series.isin(NULL_LIKE))





In [12]:
def parse_date(series):
    series=series.astype("object")
    try:
        return pd.to_datetime(series,errors="coerce",format="mixed")
    except(ValueError,TypeError):
        return pd.to_datetime(series,errors="coerce")
def to_numeric(series):
    cleaned=series.astype("string").str.replace(r"[,$\s]","",regex=True)
    return pd.to_numeric(cleaned.astype(object),errors="coerce").astype("float64")
def reject(df,mask,reason,rejected_parts):
    if mask.any():

        bad=df[mask].copy()
        bad["rejected_reason"]=reason
        rejected_parts.append(bad)
        log.warning("Rejected %d rows due to  %s",len(bad),reason)
    return df[~mask]
    

In [13]:
# transformation part
def transform(df):
    log.info("Performing the transformation")
    df=df.copy()
    rejected_parts=[]
    for col in COLUMNS:
        df[col]=clean_text(df[col])
    # email cleaning
    valid_email=df["customer_email"].str.match(EMAIL_REGEX).fillna(False).astype(bool)
    bad_email= df["customer_email"].notna() & ~valid_email
    if bad_email.any():
        log.warning("%d malformed emails set to NULL (rows kept)", bad_email.sum())
        df.loc[bad_email, "customer_email"] = pd.NA 
    # cleaning categorial values
    df["customer_city"] = df["customer_city"].str.title()
    df["category"] = df["category"].str.title()
    for col in ["store_code", "product_sku", "transaction_type"]:
        df[col] = df[col].str.upper()
    # date cleaning
    df["order_date"]=parse_date(df["order_date"])
    future_date=df["order_date"]>pd.Timestamp.today().normalize()
    if future_date.any():
        log.warning("%d future-dated rows: date set to NULL", future_date.sum())
        df.loc[future_date, "order_date"] = pd.NaT
    log.info("Rows with missing/invalid order_date (kept, date NULL): %d", df["order_date"].isna().sum())
    for col in INT_COLS + MONEY_COLS:
        df[col]= to_numeric(df[col])
    for col in ZERO_FILL_COLS :
        df[col]=df[col].fillna(0.0)
    df["net_revenue"]=(df["quantity"] * df["unit_price"]- df["store_discount"] - df["payment_gateway_discount"])

    stats={"raw_rows":len(df),"raw_net_revenue":round(df["net_revenue"].sum(),2)}

    #rejecting the corrupted rows
    df=reject(df,df["order_id"].isna() | df["order_line_item"].isna() | df["order_line_item"]%1!=0,"missing/invalid order key", rejected_parts)
    df = reject(df, df["quantity"] < 0, "negative quantity", rejected_parts)
    df = reject(df, df["quantity"].isna() | (df["quantity"] % 1 != 0),"missing/non-integer quantity", rejected_parts)
    df = reject(df, df["unit_price"].isna() | (df["unit_price"] < 0),"missing/negative unit_price", rejected_parts)
    df=reject(df,(df[ZERO_FILL_COLS]<0).any(axis=1),"negative discount/tax", rejected_parts)
    df = reject(df, df["amount_paid"].isna(), "missing amount_paid", rejected_parts)
    too_long=pd.Series(False,index=df.index)
    for col,limit in MAX_LEN.items():
        too_long |= df[col].str.len()>limit
    df=reject(df,too_long.fillna(False).astype(bool),"value longer than column size", rejected_parts)

    #removing duplicates
    key=["order_id","order_line_item"]
    exact_dups=df.duplicated(subset=COLUMNS,keep="first").sum()
    key_dups=df.duplicated(subset=key,keep="last")
    log.info("Duplicates on key: %d (exact copies: %d, same key but different values: %d)",key_dups.sum(), exact_dups, key_dups.sum()- exact_dups)
    df=reject(df,key_dups,"duplicate order_id + order_line_item", rejected_parts)

    expected = (df["quantity"] * df["unit_price"] - df["store_discount"]- df["payment_gateway_discount"] + df["tax_amount"])
    suspect = (expected - df["amount_paid"]).abs() > AMOUNT_TOLERANCE
    stats["suspect_rows"] = int(suspect.sum())
    if suspect.any():
        log.warning("%d rows where amount_paid != computed amount (loaded, saved to %s)", suspect.sum(), suspect_file)
        df[suspect][COLUMNS].to_csv(suspect_file, index=False)
    for col in INT_COLS:
        df[col] = df[col].astype("int64")
    for col in MONEY_COLS:
        df[col] = df[col].round(2)
    
    if rejected_parts:
        rejected=pd.concat(rejected_parts)
        rejected.drop(columns=["net_revenue"]).to_csv(reject_file, index=False)
        log.info("Rejected rows written to %s", reject_file)
        stats["rejected_rows"] = len(rejected)
        stats["rejected_net_revenue"] = round(rejected["net_revenue"].sum(), 2)
    else:
        stats["rejected_rows"], stats["rejected_net_revenue"] = 0, 0.0
    
    stats["clean_rows"] = len(df)
    stats["clean_net_revenue"] = round(df["net_revenue"].sum(), 2)
    clean = df[COLUMNS].reset_index(drop=True)
    log.info("Transform done: %d raw -> %d clean, %d rejected",stats["raw_rows"], stats["clean_rows"], stats["rejected_rows"])
    return clean, stats




In [14]:
DDL=[
    '''CREATE TABLE IF NOT EXISTS dim_product(
    product_sku VARCHAR(50) PRIMARY KEY,
           category    VARCHAR(50)
    )ENGINE=InnoDB''',
    '''CREATE TABLE IF NOT EXISTS dim_store (
           store_code VARCHAR(50) PRIMARY KEY,
           city       VARCHAR(100)
       ) ENGINE=InnoDB''',
    '''CREATE TABLE IF NOT EXISTS dim_customer (
           customer_email   VARCHAR(255) PRIMARY KEY,
           first_order_date DATE
       ) ENGINE=InnoDB''',
       '''CREATE TABLE IF NOT EXISTS fact_retail_sales (
           order_id                 VARCHAR(50)   NOT NULL,
           order_line_item          INT           NOT NULL,
           order_date               DATE          NULL,
           customer_email           VARCHAR(255)  NULL,
           customer_city            VARCHAR(100)  NULL,
           store_code               VARCHAR(50)   NULL,
           product_sku              VARCHAR(50)   NULL,
           category                 VARCHAR(50)   NULL,
           quantity                 INT           NOT NULL,
           unit_price               DECIMAL(10,2) NOT NULL,
           store_discount           DECIMAL(10,2) NOT NULL DEFAULT 0,
           payment_gateway_discount DECIMAL(10,2) NOT NULL DEFAULT 0,
           tax_amount               DECIMAL(10,2) NOT NULL DEFAULT 0,
           amount_paid              DECIMAL(10,2) NOT NULL,
           transaction_type         VARCHAR(20)   NULL,
           net_revenue DECIMAL(12,2) GENERATED ALWAYS AS
               (quantity * unit_price - store_discount - payment_gateway_discount) STORED,
           loaded_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
           PRIMARY KEY (order_id, order_line_item),
           CONSTRAINT chk_quantity CHECK (quantity >= 0),
           CONSTRAINT chk_unit_price CHECK (unit_price >= 0),
           INDEX idx_fact_order_date (order_date),
           INDEX idx_fact_store (store_code),
           INDEX idx_fact_product (product_sku),
           INDEX idx_fact_customer (customer_email)
       ) ENGINE=InnoDB''',
 
    '''CREATE TABLE IF NOT EXISTS stg_retail_sales (
           order_id VARCHAR(50), order_line_item INT, order_date DATE, customer_email VARCHAR(255),
           customer_city VARCHAR(100), store_code VARCHAR(50), product_sku VARCHAR(50),
           category VARCHAR(50), quantity INT, unit_price DECIMAL(10,2), store_discount DECIMAL(10,2),
           payment_gateway_discount DECIMAL(10,2), tax_amount DECIMAL(10,2),
           amount_paid DECIMAL(10,2), transaction_type VARCHAR(20)
       ) ENGINE=InnoDB''',
       '''CREATE OR REPLACE VIEW vw_sales_monthly AS
       SELECT DATE_SUB(order_date, INTERVAL DAYOFMONTH(order_date) - 1 DAY) AS sales_month,
              category,
              COUNT(DISTINCT order_id) AS orders,
              SUM(quantity)            AS units,
              SUM(net_revenue)         AS net_revenue
       FROM fact_retail_sales
       WHERE order_date IS NOT NULL
       GROUP BY sales_month, category'''
]

In [15]:

COL_LIST = ", ".join(COLUMNS)
NON_KEY = [c for c in COLUMNS if c not in ("order_id", "order_line_item")]


In [16]:
UPSERT_FACT = f"""
INSERT INTO fact_retail_sales ({COL_LIST})
SELECT {COL_LIST} FROM stg_retail_sales
ON DUPLICATE KEY UPDATE
    {", ".join(f"{c} = VALUES({c})" for c in NON_KEY)}
"""

In [17]:
UPSERT_DIMS = [
    """INSERT INTO dim_store (store_code, city)
       SELECT store_code, customer_city FROM (
           SELECT store_code, customer_city,
                  ROW_NUMBER() OVER (PARTITION BY store_code
                                     ORDER BY order_date IS NULL, order_date DESC) AS rn
           FROM stg_retail_sales WHERE store_code IS NOT NULL
       ) AS ranked WHERE rn = 1
       ON DUPLICATE KEY UPDATE city = VALUES(city)""",
    """INSERT INTO dim_product (product_sku, category)
       SELECT product_sku, category FROM (
           SELECT product_sku, category,
                  ROW_NUMBER() OVER (PARTITION BY product_sku
                                     ORDER BY order_date IS NULL, order_date DESC) AS rn
           FROM stg_retail_sales WHERE product_sku IS NOT NULL
       ) AS ranked WHERE rn = 1
       ON DUPLICATE KEY UPDATE category = VALUES(category)""",
    """INSERT INTO dim_customer (customer_email, first_order_date)
       SELECT customer_email, first_order_date FROM (
           SELECT customer_email, MIN(order_date) AS first_order_date
           FROM stg_retail_sales WHERE customer_email IS NOT NULL GROUP BY customer_email
       ) AS c
       ON DUPLICATE KEY UPDATE first_order_date = LEAST(
           COALESCE(dim_customer.first_order_date, VALUES(first_order_date)),
           COALESCE(VALUES(first_order_date), dim_customer.first_order_date))""",
]

In [18]:

def df_to_rows(df):
    out=df[COLUMNS].copy()
    out["order_date"]=df["order_date"].dt.date
    out=out.astype(object).where(out.notna(),None)
    return list(out.itertuples(index=False,name=None))
 

In [19]:
def run_reconciliation(cursor,stats):

    cursor.execute("""SELECT COUNT(*),COALESCE(SUM(f.net_revenue),0) FROM fact_retail_sales f JOIN stg_retail_sales s USING (order_id, order_line_item)""")
    db_rows,db_revenue=cursor.fetchone()
    db_revenue=round(float(db_revenue), 2)
    accounted = stats["clean_rows"] + stats["rejected_rows"]
    revenue_accounted = round(stats["clean_net_revenue"] + stats["rejected_net_revenue"], 2)
 
    print("\n=========== RECONCILIATION AUDIT ===========")
    print(f"{'Source rows (raw CSV)':38}: {stats['raw_rows']:>14,}")
    print(f"{'  - rejected / duplicates':38}: {stats['rejected_rows']:>14,}")
    print(f"{'  = expected in warehouse':38}: {stats['clean_rows']:>14,}")
    print(f"{'Rows found in warehouse':38}: {db_rows:>14,}")
    print(f"{'Source net revenue (raw)':38}: {stats['raw_net_revenue']:>14,.2f}")
    print(f"{'  - rejected net revenue':38}: {stats['rejected_net_revenue']:>14,.2f}")
    print(f"{'  = expected net revenue':38}: {stats['clean_net_revenue']:>14,.2f}")
    print(f"{'Warehouse net revenue':38}: {db_revenue:>14,.2f}")
    print(f"{'Suspect rows (amount mismatch)':38}: {stats.get('suspect_rows', 0):>14,}")
 
    checks = {
        "row count: raw == loaded + rejected": stats["raw_rows"] == accounted,
        "row count: expected == warehouse": stats["clean_rows"] == db_rows,
        "revenue: raw == loaded + rejected": abs(stats["raw_net_revenue"] - revenue_accounted) < 0.05,
        "revenue: expected == warehouse": abs(stats["clean_net_revenue"] - db_revenue) < 0.05,
    }
    for name, ok in checks.items():
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
 
    cursor.execute("""
        SELECT COUNT(*),
               COALESCE(SUM(order_date IS NULL), 0),
               COALESCE(SUM(customer_email IS NULL), 0),
               MIN(order_date), MAX(order_date)
        FROM fact_retail_sales
    """)
    total, null_dates, null_emails, min_d, max_d = cursor.fetchone()
    print(f"Fact table total rows: {total:,} | NULL dates: {int(null_dates):,} | "
          f"NULL emails: {int(null_emails):,} | date range: {min_d} -> {max_d}")
    print("============================================\n")
 
    if not all(checks.values()):
        raise RuntimeError("Reconciliation failed - rolling back the load")
    



In [20]:
def print_insights(cursor):
    """A few business insights straight from the warehouse."""
    def show(title, sql, fmt):
        print(f"\n--- {title} ---")
        cursor.execute(sql)
        for row in cursor.fetchall():
            print(fmt.format(*row))
 
    show("Overall KPIs",
         """SELECT COUNT(DISTINCT order_id), COUNT(DISTINCT customer_email),
                   SUM(net_revenue), SUM(net_revenue) / NULLIF(COUNT(DISTINCT order_id), 0)
            FROM fact_retail_sales""",
         "Orders: {:,} | Customers: {:,} | Net revenue: {:,.2f} | Avg order value: {:,.2f}")
    show("Top 5 cities by net revenue",
         """SELECT customer_city, SUM(net_revenue) FROM fact_retail_sales
            GROUP BY customer_city ORDER BY 2 DESC LIMIT 5""",
         "{:<20} {:>16,.2f}")
    show("Revenue share by category",
         """SELECT category, SUM(net_revenue),
                   ROUND(100 * SUM(net_revenue) / NULLIF(SUM(SUM(net_revenue)) OVER (), 0), 1)
            FROM fact_retail_sales GROUP BY category ORDER BY 2 DESC""",
         "{:<20} {:>16,.2f}  ({}%)")
    show("Net revenue by year",
         """SELECT YEAR(order_date), SUM(net_revenue) FROM fact_retail_sales
            WHERE order_date IS NOT NULL GROUP BY YEAR(order_date) ORDER BY 1""",
         "{}  {:>16,.2f}")
    show("Discount rate by store (store + gateway discounts as % of gross sales)",
         """SELECT store_code,
                   ROUND(100 * SUM(store_discount + payment_gateway_discount)
                         / NULLIF(SUM(quantity * unit_price), 0), 2)
            FROM fact_retail_sales GROUP BY store_code ORDER BY 2 DESC LIMIT 5""",
         "{:<14} {}%")
 

In [21]:
def load_and_reconcile_data(df,stats):
    log.info("loading the info into mySQL database")
    ensure_database()
    conn=get_db_connection()
    cursor=None
    try:

        cursor=conn.cursor(buffered=True)
        for statement in DDL:
            cursor.execute(statement)
        conn.commit()
        cursor.execute("DELETE FROM stg_retail_sales")
        rows=df_to_rows(df)
        placeholders=", ".join(["%s"]*len(COLUMNS))
        insert_sql=f"INSERT INTO stg_retail_sales({COL_LIST}) values ({placeholders})"
        for start in range(0,len(rows),BATCH_SIZE):
            cursor.executemany(insert_sql,rows[start:start+BATCH_SIZE])
       
        for sql in UPSERT_DIMS:
            cursor.execute(sql)
        cursor.execute(UPSERT_FACT)
        log.info("Running reconciliation audit...")
        run_reconciliation(cursor, stats)
 
        conn.commit()                   
        log.info("Data loaded successfully.")
 
        print_insights(cursor)         
    except Exception as e:
        conn.rollback()                 
        log.error("ETL Pipeline failed: %s", e)
        raise
    finally:
        if cursor:
            cursor.close()
        conn.close()
        



In [22]:
raw_df = extract_data("store_sales.dat")
clean_df, run_stats = transform(raw_df)
load_and_reconcile_data(clean_df, run_stats)

2026-10-25 18:42:16,186 [INFO] retail_etl: Extracted 25000 rows
2026-10-25 18:42:16,189 [INFO] retail_etl: Performing the transformation
2026-10-25 18:42:16,241 [INFO] retail_etl: Rows with missing/invalid order_date (kept, date NULL): 0
2026-10-25 18:42:16,324 [WARNING] retail_etl: Rejected 505 rows due to  negative discount/tax
2026-10-25 18:42:16,355 [INFO] retail_etl: Duplicates on key: 339 (exact copies: 339, same key but different values: 0)
2026-10-25 18:42:16,359 [WARNING] retail_etl: Rejected 339 rows due to  duplicate order_id + order_line_item
2026-10-25 18:42:16,366 [WARNING] retail_etl: 534 rows where amount_paid != computed amount (loaded, saved to suspect_rows.csv)
2026-10-25 18:42:16,388 [INFO] retail_etl: Rejected rows written to rejected_rows.csv
2026-10-25 18:42:16,392 [INFO] retail_etl: Transform done: 25000 raw -> 24156 clean, 844 rejected
2026-10-25 18:42:16,393 [INFO] retail_etl: loading the info into mySQL database
2026-10-25 18:42:20,056 [INFO] retail_etl: Runn


=========== RECONCILIATION AUDIT ===========
Source rows (raw CSV)                 :         25,000
  - rejected / duplicates             :            844
  = expected in warehouse             :         24,156
Rows found in warehouse               :         24,156
Source net revenue (raw)              :  15,029,799.08
  - rejected net revenue              :     447,028.09
  = expected net revenue              :  14,582,770.99
Warehouse net revenue                 :  14,582,770.99
Suspect rows (amount mismatch)        :            534
  [PASS] row count: raw == loaded + rejected
  [PASS] row count: expected == warehouse
  [PASS] revenue: raw == loaded + rejected
  [PASS] revenue: expected == warehouse
Fact table total rows: 24,156 | NULL dates: 0 | NULL emails: 0 | date range: 2021-01-01 -> 2026-06-01


--- Overall KPIs ---
Orders: 14,015 | Customers: 5,000 | Net revenue: 14,582,770.99 | Avg order value: 1,040.51

--- Top 5 cities by net revenue ---
Denver                   1,900,571.1

In [23]:
print("Unique customers:", clean_df['customer_email'].nunique())
print(clean_df['customer_email'].value_counts().head(5))

Unique customers: 5000
customer_email
customer_84@email.com     9
customer_220@email.com    9
customer_251@email.com    9
customer_336@email.com    9
customer_553@email.com    9
Name: count, dtype: int64[pyarrow]
